
# SVM Baseline for Malware Detection

## Objectives

The objective of this notebook is to develop a Support Vector Machine (SVM) baseline model for malware detection using the preprocessed BODMAS dataset.

The model classifies Windows executable files into two categories:
- 0: Benign
- 1: Malicious

The main objectives are:
- Load and validate the dataset.
- Preprocess the input features.
- Build an SVM classification model.
- Experiment with different hyperparameters.
- Evaluate the model using classification metrics.
- Save the trained model for future adversarial experiments.


## 1. Data Integration

### 1.1 Loading the Preprocessed Dataset

The preprocessed BODMAS dataset is loaded from the processed data directory.

The dataset has already been divided into training and testing subsets
during the preprocessing stage.

Each sample is represented by 2,381 numerical features.

Before proceeding with model development, we verify:

- The dimensions of the training and testing datasets.
- The consistency between features and labels.
- The presence of missing or infinite values.
- The distribution of the target classes.

The training set will be used for model development, while the test set
will remain reserved for the final evaluation.

In [11]:
from pathlib import Path
import numpy as np
import joblib

# Path to the preprocessed dataset
DATA_DIR = Path('../../data/processed/bodmas')

X_train = np.load(DATA_DIR / 'X_train.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')

X_test = np.load(DATA_DIR / 'X_test.npy')
y_test = np.load(DATA_DIR / 'y_test.npy')

print("Training features:", X_train.shape)
print("Training labels:", y_train.shape)

print("Testing features:", X_test.shape)
print("Testing labels:", y_test.shape)

Training features: (107548, 2381)
Training labels: (107548,)
Testing features: (26887, 2381)
Testing labels: (26887,)


### 1.2 Dataset Integrity and Shape

Dataset integrity checks are performed to ensure that the
loaded data is valid and suitable for SVM classification.

The verification includes:

- Consistency between feature matrices and label vectors.
- Consistency of feature dimensions across both data splits.
- Detection of missing and infinite values.
- Verification of the binary target labels.

These checks ensure that the input data satisfies the
requirements of the classification pipeline.

In [12]:
# Check dataset dimensions
assert X_train.shape[0] == len(y_train)
assert X_test.shape[0] == len(y_test)

# Check feature consistency
assert X_train.shape[1] == X_test.shape[1]

# Check missing and infinite values
print("NaN values in training:", np.isnan(X_train).sum())
print("NaN values in testing:", np.isnan(X_test).sum())

print("Infinite values in training:", np.isinf(X_train).sum())
print("Infinite values in testing:", np.isinf(X_test).sum())

# Check label values
print("Training labels:", np.unique(y_train))
print("Testing labels:", np.unique(y_test))

# Check class distribution
print("\nTraining class distribution:")
print(np.unique(y_train, return_counts=True))

print("\nTesting class distribution:")
print(np.unique(y_test, return_counts=True))

# Verify that the data is valid
assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()

print("\nData validation completed successfully.")

NaN values in training: 0
NaN values in testing: 0
Infinite values in training: 0
Infinite values in testing: 0
Training labels: [0 1]
Testing labels: [0 1]

Training class distribution:
(array([0, 1], dtype=int32), array([61714, 45834]))

Testing class distribution:
(array([0, 1], dtype=int32), array([15428, 11459]))

Data validation completed successfully.


### 1.3 Label Distribution

The distribution of benign and malicious samples is examined
to identify potential class imbalance.

This analysis is important because an imbalanced dataset may
affect the learning process and classification performance.

The class distribution is examined separately for the training
and testing sets to verify the consistency of the data splits.

In [20]:
import pandas as pd

train_distribution = pd.Series(y_train).value_counts().sort_index()
test_distribution = pd.Series(y_test).value_counts().sort_index()

distribution = pd.DataFrame({
    "Training Set": train_distribution,
    "Testing Set": test_distribution
})

distribution.index = ["Benign", "Malicious"]

display(distribution)

,Training Set,Testing Set
Benign,61714,15428
Malicious,45834,11459


## 2. Feature Selection

### 2.1 Feature Selection Strategy

Feature selection aims to identify the most informative
features while reducing the dimensionality of the input data.

The ANOVA F-test is selected as a statistical feature-ranking
method. It evaluates the relationship between each individual
feature and the target class.

The SelectKBest method is used to retain the highest-ranking
features according to their F-scores.

Different feature subset sizes will be investigated to assess
their impact on classification performance and computational cost.

Feature selection will be integrated into the SVM pipeline
to ensure that feature ranking is learned exclusively from
the training data during cross-validation.

2.2 Feature Selection Configuration

In [18]:
from sklearn.feature_selection import SelectKBest, f_classif

# Number of available features
n_features = X_train.shape[1]

# Candidate numbers of selected features
k_values = [100, 300, 600, 1000, 'all']

# Keep only valid values
k_values = [
    k for k in k_values
    if k == 'all' or k <= n_features
]

print("Total number of features:", n_features)
print("Feature selection candidates:", k_values)

Total number of features: 2381
Feature selection candidates: [100, 300, 600, 1000, 'all']


## 3. SVM Model Development

### 3.1 LinearSVC Classifier

Support Vector Machine (SVM) is a supervised learning
algorithm widely used for classification tasks.

It identifies an optimal decision boundary, known as a
hyperplane, to separate samples belonging to different classes.

For malware classification, LinearSVC is selected as the
baseline classifier.

This implementation is particularly suitable for high-dimensional
classification problems and offers a computationally efficient
approach for large-scale linear classification.

The classifier will be used to distinguish between benign
and malicious software samples.

### 3.2 SVM Pipeline Configuration

A Scikit-learn Pipeline is established to combine feature
selection and SVM classification into a single workflow.

The pipeline consists of two components:

- SelectKBest: selects the most informative features using
  the ANOVA F-test.
- LinearSVC: performs the final binary classification.

The classifier uses balanced class weights to account for
potential differences in class frequencies.

The maximum number of iterations is increased to provide
sufficient optimization time for convergence.

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectKBest, f_classif

# Define the SVM pipeline
svm_pipeline = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('svm', LinearSVC(
        class_weight='balanced',
        max_iter=10000,
        random_state=42
    ))
])

print("SVM Pipeline:")
print(svm_pipeline)

SVM Pipeline:
Pipeline(steps=[('feature_selection', SelectKBest()),
                ('svm',
                 LinearSVC(class_weight='balanced', max_iter=10000,
                           random_state=42))])
